# Image → 3D Matrix
Given a multi-frame TIFF address, generate cell geometry as a **3D** matrix.  
Steps:
1. Load the TIFF stack (Z × Y × X or Y × X per-frame).
2. Normalise raw uint16 values → refractive-index (RI) values.
3. Apply 3-D Fourier low-pass + median + background-suppression filters.
4. Threshold with Otsu → binary volume mask.
5. Apply mask → extract cell-body RI volume.
6. Save as `.mat` file.

In [ ]:
# Import libraries
import numpy as np
from PIL import Image
import tifffile as tf
import scipy as sp
import os
import matplotlib.pyplot as plt

# Filters
from scipy.ndimage import median_filter, gaussian_filter
from skimage.filters import threshold_otsu

## I/O Path Allocations
Set `repo_path`, `access_address`, and `export_address` for your environment.

In [ ]:
# Path variables
repo_path      = "/path/to/repository/"   # <-- replace
export_address = "data/extracted_ri/"
access_address = "data/processed/"

def output_address(subfolder="data/", repository=repo_path):
    return f"{repository}{subfolder}"

def modified_output_address(file_name):
    name_part = file_name.split(".")[0]
    return f"{name_part}_extracted_3d.mat"

In [ ]:
# Specify file
file_name   = "63_HT3D.tiff"   # <-- replace with actual file name
access_path = f"{repo_path}{access_address}{file_name}"
print(f"Accessing: {access_path}")

## Load TIFF stack

**2-D vs 3-D handling**
- `tifffile.imread` returns shape `(Z, Y, X)` for a standard multi-frame TIFF.
- If your file stores frames differently (e.g., `(T, Z, Y, X)`) adjust the squeeze/indexing below.
- A single-frame TIFF yields `(Y, X)`; we expand it to `(1, Y, X)` automatically.

In [ ]:
ext = file_name.split(".")[-1].lower()

if ext in ("tiff", "tif"):
    raw = tf.imread(access_path)
    print("TIFF loaded successfully!")
elif ext in ("jpeg", "jpg", "png"):
    raw = np.array(Image.open(access_path))
    print("Image loaded successfully!")
else:
    raise ValueError("Unsupported file format. Use TIFF, JPEG, or PNG.")

raw_mat = np.array(raw, dtype=np.float64)

# -----------------------------------------------------------------
# Ensure the array is 3-D: (Z, Y, X)
# -----------------------------------------------------------------
if raw_mat.ndim == 2:
    # Single-slice image — promote to (1, Y, X)
    raw_mat = raw_mat[np.newaxis, ...]
    print("Single-frame TIFF detected; treated as 1 Z-slice.")
elif raw_mat.ndim == 4:
    # e.g., (T, Z, Y, X) — take first time-point
    raw_mat = raw_mat[0]
    print(f"4-D array detected; using first time-point. New shape: {raw_mat.shape}")
elif raw_mat.ndim != 3:
    raise ValueError(f"Unexpected array dimensionality: {raw_mat.ndim}D")

print(f"Raw shape (Z, Y, X): {raw_mat.shape}  |  dtype: {raw_mat.dtype}")

# Normalise to RI values
ri_mat = raw_mat / 10000.0
print(f"RI range: [{ri_mat.min():.4f}, {ri_mat.max():.4f}]")

In [ ]:
# Quick sanity-check: show the middle Z-slice
mid_z = ri_mat.shape[0] // 2
plt.figure(figsize=(7, 5))
plt.imshow(ri_mat[mid_z], cmap="viridis")
plt.colorbar(label="RI")
plt.title(f"Middle Z-slice (z={mid_z}) — raw RI")
plt.tight_layout()
plt.show()

## 3-D Filtering
1. **3-D Fourier low-pass** — removes speckle noise in all three spatial dimensions.
2. **3-D median filter** — further smoothing.
3. **3-D Gaussian background suppression** — flattens the illumination gradient.

In [ ]:
def fourier_lowpass_3d(ri, sigma=40):
    """3-D Gaussian low-pass filter in the frequency domain."""
    F       = np.fft.fftn(ri)
    F_shift = np.fft.fftshift(F)

    nz, ny, nx = ri.shape
    cz, cy, cx = nz // 2, ny // 2, nx // 2

    w = np.arange(nz) - cz
    u = np.arange(ny) - cy
    v = np.arange(nx) - cx
    W, U, V = np.meshgrid(w, u, v, indexing='ij')
    D = np.sqrt(W**2 + U**2 + V**2)

    H = np.exp(-(D**2) / (2 * sigma**2))

    F_filtered = F_shift * H
    F_ishift   = np.fft.ifftshift(F_filtered)
    ri_filtered = np.fft.ifftn(F_ishift)
    return np.real(ri_filtered)


def evaluate_filtering_3d(ri, ri_filtered):
    """Report energy loss and RI change statistics."""
    F_orig = np.fft.fftshift(np.fft.fftn(ri))
    F_filt = np.fft.fftshift(np.fft.fftn(ri_filtered))
    E_orig = np.sum(np.abs(F_orig)**2)
    E_filt = np.sum(np.abs(F_filt)**2)
    diff   = ri - ri_filtered
    print(f"Energy loss       : {(E_orig - E_filt) / E_orig:.6f}")
    print(f"Mean |change|     : {np.mean(np.abs(diff)):.6f}")
    print(f"Max  |change|     : {np.max(np.abs(diff)):.6f}")
    print(f"Peak RI (before)  : {ri.max():.6f}")
    print(f"Peak RI (after)   : {ri_filtered.max():.6f}")

In [ ]:
# --- 3-D Fourier low-pass ---
print("Applying 3-D Fourier low-pass filter...")
# Sigma scales with the smallest spatial dimension so large stacks are not over-smoothed
sigma_fourier = min(ri_mat.shape) // 8
fourier_vol   = fourier_lowpass_3d(ri_mat, sigma=sigma_fourier)

# --- 3-D Median filter ---
print("Applying 3-D median filter...")
ri_med = median_filter(fourier_vol, size=3)

# --- 3-D background suppression via Gaussian ---
print("Suppressing background...")
background = gaussian_filter(ri_med, sigma=50)
ri_flat    = ri_med - background

# Visualise middle slice after filtering
mid_z = ri_flat.shape[0] // 2
plt.figure(figsize=(7, 5))
plt.imshow(ri_flat[mid_z], cmap="viridis")
plt.colorbar(label="RI (filtered)")
plt.title(f"Middle Z-slice (z={mid_z}) — filtered RI")
plt.tight_layout()
plt.show()

evaluate_filtering_3d(ri_mat, ri_flat)

## 3-D Mask Extraction
1. Compute a single Otsu threshold across the entire 3-D volume.
2. Build a binary mask (voxel ≥ threshold → 1, else 0).
3. Apply mask to the RI volume.

In [ ]:
# Otsu threshold on the full 3-D volume
ri_threshold = threshold_otsu(ri_flat)
binary_vol   = (ri_flat >= ri_threshold).astype(np.uint8)

print(f"Otsu threshold   : {ri_threshold:.6f}")
print(f"Binary volume shape (Z, Y, X): {binary_vol.shape}")
print(f"Non-zero voxels  : {binary_vol.sum()} / {binary_vol.size}")

# Visualise mask at middle slice
mid_z = binary_vol.shape[0] // 2
plt.figure(figsize=(7, 5))
plt.imshow(binary_vol[mid_z], cmap="gray")
plt.colorbar(label="Mask")
plt.title(f"Middle Z-slice (z={mid_z}) — binary mask")
plt.tight_layout()
plt.show()

In [ ]:
# Sanity check: shapes must match
if ri_mat.shape != binary_vol.shape:
    raise ValueError(
        f"Shape mismatch: RI {ri_mat.shape} vs mask {binary_vol.shape}")

# Apply mask — zero out background
masked_ri_vol = binary_vol * ri_mat

print(f"Masked RI volume shape : {masked_ri_vol.shape}")
print(f"Max RI in cell body    : {masked_ri_vol.max():.6f}")

# Visualise result at middle slice
mid_z = masked_ri_vol.shape[0] // 2
plt.figure(figsize=(7, 5))
plt.imshow(masked_ri_vol[mid_z], cmap="viridis")
plt.colorbar(label="Masked RI")
plt.title(f"Middle Z-slice (z={mid_z}) — masked RI")
plt.tight_layout()
plt.show()

## Saving

In [ ]:
output_path = output_address(export_address, repo_path)
output_name = modified_output_address(file_name)

try:
    sp.io.savemat(
        f"{output_path}{output_name}",
        {"masked_ri": masked_ri_vol}
    )
    print(f"3-D extracted RI volume saved → {output_path}{output_name}")
except Exception as e:
    print(f"Error saving: {e}")
    raise